# Sesión 3 - Práctica 3: Sistema Cooperativo de Limpieza (EN VIVO)

## Objetivos
- Ver un ejemplo COMPLETO de sistema multiagente COOPERATIVO
- Reutilizar código de Sesión 2 (GridEnvironment)
- Implementar Coordinator Agent que asigna zonas
- Implementar Cleaner Agents que se coordinan
- Comunicación FIPA-ACL para cooperación
- Visualización paso a paso del proceso

## DIFERENCIA CON ACTIVIDAD 1

**Este notebook:** Sistema COOPERATIVO de limpieza  
- Robots colaboran  
- Objetivo común  

**Actividad 1:** Sistema COMPETITIVO de juegos  
- Agentes compiten  
- Objetivos opuestos  

**Misma arquitectura Manager-Workers, diferente dominio.**

## Parte 1: Grid Environment (YA CONOCIDO)

In [ ]:
import numpy as np
import random
import time

class GridEnvironment:
    def __init__(self, width=5, height=5, dirt_probability=0.3):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)
        
        for i in range(height):
            for j in range(width):
                if random.random() < dirt_probability:
                    self.grid[i][j] = 1
    
    def is_dirty(self, x, y):
        if 0 <= x < self.width and 0 <= y < self.height:
            return self.grid[y][x] == 1
        return False
    
    def clean(self, x, y):
        if 0 <= x < self.width and 0 <= y < self.height:
            if self.grid[y][x] == 1:
                self.grid[y][x] = 0
                return True
        return False
    
    def count_dirty_cells(self):
        return np.sum(self.grid)
    
    def is_clean(self):
        return self.count_dirty_cells() == 0
    
    def display(self, robot_positions=None, title="Grid"):
        print(f"\n{title}:")
        print("  ", end="")
        for j in range(self.width):
            print(f"{j} ", end="")
        print()
        
        for i in range(self.height):
            print(f"{i} ", end="")
            for j in range(self.width):
                if robot_positions and (j, i) in robot_positions:
                    robot_num = robot_positions.index((j, i)) + 1
                    print(f"R{robot_num}", end=" ")
                elif self.grid[i][j] == 1:
                    print("X ", end="")
                else:
                    print(". ", end="")
            print()
        print(f"Celdas sucias: {self.count_dirty_cells()}")

print("GridEnvironment definido.")

## Parte 2: Cleaner Agent

### GAP 1: Responder AGREE al Coordinator

In [ ]:
from spade import agent
from spade.behaviour import CyclicBehaviour
from spade.message import Message
import asyncio
import json

class CleanerAgent(agent.Agent):
    def __init__(self, jid, password, robot_name, robot_number, environment, coordinator_jid):
        super().__init__(jid, password)
        self.robot_name = robot_name
        self.robot_number = robot_number
        self.environment = environment
        self.coordinator_jid = coordinator_jid
        self.assigned_zone = None
        self.cells_cleaned = 0
        self.position = (0, 0)
    
    class CleanBehaviour(CyclicBehaviour):
        async def run(self):
            msg = await self.receive(timeout=5)
            
            if msg:
                performative = msg.get_metadata('performative')
                
                if performative == 'request':
                    data = json.loads(msg.body)
                    
                    if data['action'] == 'clean_zone':
                        self.agent.assigned_zone = data['zone']
                        
                        # TODO: Crear mensaje AGREE
                        
                        
                        # TODO: Set performative 'agree'
                        
                        
                        # TODO: Set body con datos
                        
                        
                        # TODO: Enviar mensaje
                        
                        
                        print(f"\n[{self.agent.robot_name}] Zona asignada: {self.agent.assigned_zone}")
                        
                        # Limpiar zona
                        await self.clean_assigned_zone()
                        
                        # Reportar completado
                        inform_msg = Message(to=str(msg.sender))
                        inform_msg.set_metadata('performative', 'inform')
                        inform_msg.body = json.dumps({
                            'robot': self.agent.robot_name,
                            'status': 'completed',
                            'cells_cleaned': self.agent.cells_cleaned
                        })
                        await self.send(inform_msg)
                        
                        print(f"[{self.agent.robot_name}] ✓ Zona completada - {self.agent.cells_cleaned} celdas limpiadas")
        
        async def clean_assigned_zone(self):
            if not self.agent.assigned_zone:
                return
            
            x_start, y_start, x_end, y_end = self.agent.assigned_zone
            
            for y in range(y_start, y_end + 1):
                for x in range(x_start, x_end + 1):
                    if self.agent.environment.is_dirty(x, y):
                        await asyncio.sleep(0.3)
                        
                        if self.agent.environment.clean(x, y):
                            self.agent.cells_cleaned += 1
                            self.agent.position = (x, y)
                            
                            # Reportar progreso para visualización
                            progress_msg = Message(to=self.agent.coordinator_jid)
                            progress_msg.set_metadata('performative', 'inform')
                            progress_msg.body = json.dumps({
                                'type': 'progress',
                                'robot': self.agent.robot_name,
                                'robot_number': self.agent.robot_number,
                                'position': (x, y),
                                'cells_cleaned': self.agent.cells_cleaned
                            })
                            await self.send(progress_msg)
    
    async def setup(self):
        print(f"[{self.robot_name}] Robot limpiador listo.")
        behaviour = self.CleanBehaviour()
        self.add_behaviour(behaviour)

print("CleanerAgent definido.")

## Parte 3: Coordinator Agent

### GAP 2: Enviar REQUEST a robots

In [ ]:
from spade.behaviour import OneShotBehaviour, CyclicBehaviour

class CoordinatorAgent(agent.Agent):
    def __init__(self, jid, password, environment, cleaners_info):
        super().__init__(jid, password)
        self.environment = environment
        self.cleaners_info = cleaners_info
        self.results = {}
        self.robot_positions = {}
        self.visualization_on = True
    
    class VisualizationBehaviour(CyclicBehaviour):
        async def run(self):
            msg = await self.receive(timeout=1)
            
            if msg:
                performative = msg.get_metadata('performative')
                
                if performative == 'inform':
                    data = json.loads(msg.body)
                    
                    if data.get('type') == 'progress' and self.agent.visualization_on:
                        robot_name = data['robot']
                        robot_num = data['robot_number']
                        position = tuple(data['position'])
                        cells = data['cells_cleaned']
                        
                        self.agent.robot_positions[robot_num] = position
                        
                        print(f"\n→ {robot_name} limpiando ({position[0]},{position[1]}) - Total: {cells}")
                        
                        positions_list = [None, None, None]
                        for num, pos in self.agent.robot_positions.items():
                            positions_list[num - 1] = pos
                        
                        self.agent.environment.display(
                            robot_positions=[p for p in positions_list if p],
                            title="Estado actual"
                        )
    
    class CoordinateCleaning(OneShotBehaviour):
        async def run(self):
            print("\n" + "="*60)
            print("SISTEMA DE LIMPIEZA COOPERATIVA INICIADO")
            print("="*60)
            
            print("\nEstado inicial:")
            self.agent.environment.display(title="Antes de limpiar")
            
            initial_dirty = self.agent.environment.count_dirty_cells()
            print(f"\nTotal sucias: {initial_dirty}")
            print(f"Robots: {len(self.agent.cleaners_info)}")
            
            zones = self.divide_into_zones()
            print(f"\nDividiendo en {len(zones)} zonas...")
            
            await self.assign_zones(zones)
            
            print("\n" + "="*60)
            print("LIMPIEZA EN PROGRESO (visualización paso a paso)")
            print("="*60)
            
            await self.wait_for_completion()
            
            self.agent.visualization_on = False
            
            await asyncio.sleep(1)
            self.show_results(initial_dirty)
            
            await self.agent.stop()
        
        def divide_into_zones(self):
            num_robots = len(self.agent.cleaners_info)
            width = self.agent.environment.width
            height = self.agent.environment.height
            
            zones = []
            
            if num_robots == 2:
                mid = width // 2
                zones.append((0, 0, mid-1, height-1))
                zones.append((mid, 0, width-1, height-1))
            elif num_robots == 3:
                third = height // 3
                zones.append((0, 0, width-1, third-1))
                zones.append((0, third, width-1, 2*third-1))
                zones.append((0, 2*third, width-1, height-1))
            else:
                zones.append((0, 0, width-1, height-1))
            
            return zones
        
        async def assign_zones(self, zones):
            for i, ((jid, name, num), zone) in enumerate(zip(self.agent.cleaners_info, zones)):
                # TODO: Crear mensaje REQUEST
                
                
                # TODO: Set performative 'request'
                
                
                # TODO: Set body con acción y zona
                
                
                # TODO: Enviar
                
                
                print(f"  Zona {i+1} → {name}: {zone}")
            
            agrees = 0
            for _ in range(len(zones)):
                msg = await self.receive(timeout=5)
                if msg and msg.get_metadata('performative') == 'agree':
                    agrees += 1
            
            print(f"\n✓ {agrees}/{len(zones)} robots confirmaron.")
        
        async def wait_for_completion(self):
            num_robots = len(self.agent.cleaners_info)
            completed = 0
            
            while completed < num_robots:
                msg = await self.receive(timeout=30)
                
                if msg and msg.get_metadata('performative') == 'inform':
                    data = json.loads(msg.body)
                    
                    if data.get('status') == 'completed':
                        robot_name = data['robot']
                        cells_cleaned = data['cells_cleaned']
                        self.agent.results[robot_name] = cells_cleaned
                        completed += 1
        
        def show_results(self, initial_dirty):
            print("\n" + "="*60)
            print("LIMPIEZA COMPLETADA")
            print("="*60)
            
            print("\nEstado final:")
            self.agent.environment.display(title="Después de limpiar")
            
            final_dirty = self.agent.environment.count_dirty_cells()
            total_cleaned = initial_dirty - final_dirty
            
            print("\n" + "-"*60)
            print("ESTADÍSTICAS POR ROBOT")
            print("-"*60)
            print(f"{'Robot':<20} {'Limpiadas':>15} {'%':>10}")
            print("-"*60)
            
            for robot_name, cells in sorted(self.agent.results.items(), 
                                           key=lambda x: x[1], reverse=True):
                pct = (cells / total_cleaned * 100) if total_cleaned > 0 else 0
                print(f"{robot_name:<20} {cells:>15} {pct:>9.1f}%")
            
            print("-"*60)
            print(f"{'TOTAL':<20} {total_cleaned:>15} {100.0:>9.1f}%")
            print("="*60)
            
            efficiency = (total_cleaned / initial_dirty * 100) if initial_dirty > 0 else 100
            print(f"\nEficiencia: {efficiency:.1f}%")
            print(f"Iniciales sucias: {initial_dirty}")
            print(f"Limpiadas: {total_cleaned}")
            print(f"Aún sucias: {final_dirty}")
    
    async def setup(self):
        print("[COORDINATOR] Iniciado.")
        
        viz_behaviour = self.VisualizationBehaviour()
        self.add_behaviour(viz_behaviour)
        
        coord_behaviour = self.CoordinateCleaning()
        self.add_behaviour(coord_behaviour)

print("CoordinatorAgent definido.")

## Parte 4: Ejecutar

In [ ]:
async def run_cooperative_cleaning():
    environment = GridEnvironment(width=6, height=6, dirt_probability=0.4)
    
    coordinator_jid = "cleaning_coordinator@anon.jabberfr.org"
    
    cleaners_info = [
        ("cleaner_robot1@anon.jabberfr.org", "Robot-1", 1),
        ("cleaner_robot2@anon.jabberfr.org", "Robot-2", 2),
        ("cleaner_robot3@anon.jabberfr.org", "Robot-3", 3)
    ]
    
    robots = []
    passwords = ["spade_cleaner1", "spade_cleaner2", "spade_cleaner3"]
    
    for (jid, name, num), password in zip(cleaners_info, passwords):
        robot = CleanerAgent(jid, password, name, num, environment, coordinator_jid)
        robots.append(robot)
    
    coordinator = CoordinatorAgent(
        coordinator_jid,
        "spade_coordinator",
        environment,
        cleaners_info
    )
    
    for robot in robots:
        await robot.start()
    
    await asyncio.sleep(2)
    await coordinator.start()
    
    while coordinator.is_alive():
        await asyncio.sleep(0.5)
    
    for robot in robots:
        await robot.stop()
    
    print("\n=== SISTEMA DETENIDO ===")

print("\nIniciando limpieza cooperativa...\n")
await run_cooperative_cleaning()

## Resumen

**Has visto:**
- Sistema multiagente cooperativo
- Arquitectura Manager-Workers
- Comunicación SPADE
- Visualización en tiempo real

**Para Actividad 1:**
- Adaptar a sistema competitivo
- Cambiar limpieza → juegos
- Mantener arquitectura y SPADE